In [ ]:
import pandas as pd
import requests
import time

API_KEY = "########################################"  # ใส่ API Key ของคุณที่นี่

print("กำลังอ่านไฟล์ netflix_titles.csv...")
df = pd.read_csv('netflix_titles.csv')

missing_directors_df = df[(df['type'] == 'Movie') & (df['director'].isnull())]

print(f"พบหนังที่ไม่มีชื่อผู้กำกับ (ที่ต้องการทดสอบ): {len(missing_directors_df)} เรื่อง")

def get_movie_director_from_tmdb(title, year):

    search_url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={title}&year={year}"
    res = requests.get(search_url)

    if res.status_code == 200:
        results = res.json().get('results', [])
        if results:
            movie_id = results[0]['id']

            credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={API_KEY}"
            cred_res = requests.get(credits_url)

            if cred_res.status_code == 200:
                crew = cred_res.json().get('crew', [])

                directors = [person['name'] for person in crew if person['job'] == 'Director']

                if directors:
                    return ", ".join(directors)

    return None

print("\nกำลังดึงข้อมูลจาก TMDB เพื่อเติมส่วนที่ขาดหาย...")

for index, row in missing_directors_df.iterrows():
    title = row['title']
    year = row['release_year']

    print(f"กำลังค้นหา: {title} ({year})...", end=" ")

    found_director = get_movie_director_from_tmdb(title, year)

    if found_director:
        print(f"--> เจอแล้ว!: {found_director}")
        df.at[index, 'director'] = found_director
    else:
        print("--> ไม่พบข้อมูล")

    time.sleep(0.2)

print("\nกระบวนการเสร็จสิ้น!")
print("จำนวน Missing Values ของผู้กำกับ (ก่อนทำ): 2634")
print(f"จำนวน Missing Values ของผู้กำกับ (หลังทำ): {df['director'].isnull().sum()}")

df.to_csv('netflix_titles_cleaned.csv', index=False, encoding='utf-8-sig')
print("\nบันทึกไฟล์ 'netflix_titles_cleaned.csv' เรียบร้อยแล้ว นำไปทำ Dashboard ต่อได้เลย!")

กำลังอ่านไฟล์ netflix_titles.csv...
พบหนังที่ไม่มีชื่อผู้กำกับ (ที่ต้องการทดสอบ): 188 เรื่อง

กำลังดึงข้อมูลจาก TMDB เพื่อเติมส่วนที่ขาดหาย...
กำลังค้นหา: 9to5: The Story of a Movement (2021)... --> เจอแล้ว!: Steven Bognar, Julia Reichert
กำลังค้นหา: Bridgerton - The Afterparty (2021)... --> ไม่พบข้อมูล
กำลังค้นหา: Last Summer (2021)... --> เจอแล้ว!: Ozan Açıktan
กำลังค้นหา: Sisters on Track (2021)... --> เจอแล้ว!: Corinne van der Borch, Tone Grøttjord-Glenne
กำลังค้นหา: Headspace: Unwind Your Mind (2021)... --> ไม่พบข้อมูล
กำลังค้นหา: Bling Empire - The Afterparty (2021)... --> ไม่พบข้อมูล
กำลังค้นหา: Cobra Kai - The Afterparty (2021)... --> ไม่พบข้อมูล
กำลังค้นหา: To All the Boys: Always and Forever - The Afterparty (2021)... --> เจอแล้ว!: Michael Fimognari
กำลังค้นหา: Trese After Dark (2021)... --> เจอแล้ว!: Gio Puyat
กำลังค้นหา: Kitty Love: An Homage to Cats (2021)... --> เจอแล้ว!: Mark Verkerk
กำลังค้นหา: Sam Smith: Love Goes - Live at Abbey Road Studios (2020)... --> ไม่พบข้อมูล


In [ ]:
import pandas as pd
import requests
import time

API_KEY = "7c2559f254b3731f0af4edfea5f92146"

print("กำลังอ่านไฟล์ netflix_titles_cleaned.csv...")
df = pd.read_csv('netflix_titles_cleaned.csv')

df['tmdb_score'] = None
df['tmdb_votes'] = None

sample_df = df
print(f"กำลังจะดึงคะแนนสำหรับข้อมูล {len(sample_df)} เรื่อง...\n")

def get_tmdb_score(title, release_year, show_type):
    """ฟังก์ชันสำหรับดึงคะแนนโหวต โดยแยกประเภท Movie และ TV Show"""

    search_type = "movie" if show_type == "Movie" else "tv"

    year_param = f"&year={release_year}" if show_type == "Movie" else f"&first_air_date_year={release_year}"

    url = f"https://api.themoviedb.org/3/search/{search_type}?api_key={API_KEY}&query={title}{year_param}"

    try:
        res = requests.get(url)
        if res.status_code == 200:
            results = res.json().get('results', [])
            if results:
                score = results[0].get('vote_average')
                votes = results[0].get('vote_count')
                return score, votes
    except Exception as e:
        pass

    return None, None

for index, row in sample_df.iterrows():
    title = row['title']
    year = row['release_year']
    show_type = row['type']

    print(f"[{show_type}] {title} ({year})...", end=" ")

    score, votes = get_tmdb_score(title, year, show_type)

    if score is not None:
        print(f"--> คะแนน: {score}/10 (โหวต {votes} คน)")
        df.at[index, 'tmdb_score'] = score
        df.at[index, 'tmdb_votes'] = votes
    else:
        print("--> ไม่พบข้อมูลคะแนน")

    time.sleep(0.2)

print("\nกระบวนการเสร็จสิ้น!")
df.to_csv('netflix_with_scores.csv', index=False, encoding='utf-8-sig')
print("บันทึกไฟล์ 'netflix_with_scores.csv' เรียบร้อยแล้ว มีทั้งชื่อผู้กำกับและคะแนนรีวิว!")

Streaming output truncated to the last 5000 lines.
[Movie] Dying to Tell (2018)... --> คะแนน: 7.0/10 (โหวต 14 คน)
[TV Show] It's Bruno! (2019)... --> คะแนน: 7.034/10 (โหวต 44 คน)
[Movie] Maria (2019)... --> คะแนน: 6.028/10 (โหวต 160 คน)
[Movie] ReMastered: The Lion's Share (2019)... --> คะแนน: 6.6/10 (โหวต 23 คน)
[Movie] Saverio Raimondo: Il Satiro Parlante (2019)... --> คะแนน: 5.7/10 (โหวต 15 คน)
[Movie] See You Yesterday (2019)... --> คะแนน: 5.54/10 (โหวต 569 คน)
[TV Show] White Gold (2019)... --> ไม่พบข้อมูลคะแนน
[Movie] Good Sam (2019)... --> คะแนน: 5.975/10 (โหวต 242 คน)
[Movie] Jonaki (2018)... --> คะแนน: 4.955/10 (โหวต 11 คน)
[Movie] PATRICK (2019) (2019)... --> คะแนน: 8.0/10 (โหวต 1 คน)
[TV Show] Dennis and Gnasher Unleashed (2017)... --> คะแนน: 6.7/10 (โหวต 3 คน)
[Movie] Hamid (2018)... --> คะแนน: 0.0/10 (โหวต 0 คน)
[TV Show] Heidi, bienvenida a casa (2017)... --> คะแนน: 8.0/10 (โหวต 8 คน)
[TV Show] I Hear You (2019)... --> คะแนน: 7.7/10 (โหวต 68 คน)
[Movie] Backdraft 2 (2019)